# Qwen3-TTS voice-clone spike

Proof of concept for Cadence's voice cloning: upload a ~15 s clip, build a reusable voice "impression" (a voice-clone prompt), then speak sample lines in that voice and time each step.

**Runs on:** Google Colab or Kaggle with a GPU (a T4 is enough for the 0.6B model). It also runs on CPU, slowly.

- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** or **P100** (needs phone verification), and turn **Internet** on.

**Consent:** only use a clip from someone who has agreed to have their voice cloned. The clip is uploaded to Colab/Kaggle for this test, so delete the runtime when you're done. The real app stays local-first.

How this maps to the app: step 4 is `POST /voices`, step 5 is `POST /tts`. Record the timings in `docs/latency.md`.

## 1. Install

In [ ]:
import subprocess
import sys

# qwen-tts pins its own transformers/accelerate versions; sox is needed by one of its imports.
subprocess.run("apt-get -qq install -y sox > /dev/null 2>&1", shell=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-tts"], check=True)

import torch

print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU will be slow)")

## 2. Settings

Set `REF_TEXT` to **exactly** what is said in the clip; the model lines the transcript up with the audio. No transcript? Set `X_VECTOR_ONLY = True` (worse likeness, no transcript needed).

In [ ]:
MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"  # try "Qwen/Qwen3-TTS-12Hz-1.7B-Base" for quality
LANGUAGE = "english"

REF_TEXT = ""  # exact transcript of the clip
X_VECTOR_ONLY = False  # True = speaker embedding only, no transcript needed

SAMPLE_LINES = [
    "Yes.",
    "Give me a second.",
    "I need some water.",
    "I need to use the restroom.",
    "Better than yesterday, thanks.",
    "Hi judges, I'll be demoing Cadence today.",
]

MAX_REF_SECONDS = 20  # longer clips are trimmed; 10-15 s of clean speech is the sweet spot

## 3. Upload the reference clip

- **Colab:** run the cell and pick the file.
- **Kaggle (or anywhere else):** add the file to the notebook and set `REF_PATH` to its path.

WAV is best; MP3/M4A also work. One speaker, a quiet room, no music.

In [ ]:
import os

REF_PATH = ""  # set this on Kaggle, e.g. "/kaggle/input/voice/clip.wav"

if not REF_PATH:
    try:
        from google.colab import files

        uploaded = files.upload()
        REF_PATH = next(iter(uploaded))
    except ImportError:
        raise SystemExit("Not on Colab: add the clip to the notebook and set REF_PATH above.")

assert os.path.exists(REF_PATH), f"missing {REF_PATH}"
print("reference clip:", REF_PATH)

In [ ]:
import librosa
import numpy as np
from IPython.display import Audio, display

# Mono, trimmed, and normalized so a quiet or long recording doesn't hurt the clone.
ref_wav, ref_sr = librosa.load(REF_PATH, sr=None, mono=True)
ref_wav, _ = librosa.effects.trim(ref_wav, top_db=30)
ref_wav = ref_wav[: int(MAX_REF_SECONDS * ref_sr)]
ref_wav = (ref_wav / max(1e-6, np.abs(ref_wav).max()) * 0.9).astype(np.float32)

print(f"{len(ref_wav) / ref_sr:.1f} s at {ref_sr} Hz")
if len(ref_wav) / ref_sr < 3:
    print("WARNING: under 3 s of speech; the clone will be poor.")
display(Audio(ref_wav, rate=ref_sr))

## 4. Load the model and make the voice impression

In [ ]:
import time

import torch
from qwen_tts import Qwen3TTSModel

if torch.cuda.is_available():
    device = "cuda:0"
    # T4/P100 have no fast bfloat16, so use float16 there. If the output sounds like noise, switch to torch.float32.
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    device, dtype = "cpu", torch.float32

t = time.perf_counter()
tts = Qwen3TTSModel.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
load_s = time.perf_counter() - t
print(f"loaded {MODEL_ID} on {device} ({dtype}) in {load_s:.1f} s")
if torch.cuda.is_available():
    print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
if not X_VECTOR_ONLY and not REF_TEXT.strip():
    raise SystemExit("Set REF_TEXT to the clip's transcript (or set X_VECTOR_ONLY = True).")

t = time.perf_counter()
voice_prompt = tts.create_voice_clone_prompt(
    ref_audio=(ref_wav, ref_sr),
    ref_text=None if X_VECTOR_ONLY else REF_TEXT,
    x_vector_only_mode=X_VECTOR_ONLY,
)
clone_ms = (time.perf_counter() - t) * 1000
print(f"voice impression built in {clone_ms:.0f} ms")

# The app does this once per voice and reuses it for every line, with no re-analysis.
torch.save(voice_prompt, "voice_prompt.pt")
print("saved voice_prompt.pt")

## 5. Speak sample lines in the cloned voice

The first line includes warm-up cost (cold); later lines show the steady-state (warm) speed.

In [ ]:
import soundfile as sf

os.makedirs("out", exist_ok=True)
results = []
for i, text in enumerate(SAMPLE_LINES):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t = time.perf_counter()
    wavs, sr = tts.generate_voice_clone(text=text, language=LANGUAGE, voice_clone_prompt=voice_prompt)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t) * 1000
    audio_s = len(wavs[0]) / sr
    sf.write(f"out/{i:02d}.wav", wavs[0], sr)
    results.append(
        {
            "text": text,
            "chars": len(text),
            "gen_ms": round(ms),
            "audio_s": round(audio_s, 2),
            "realtime_factor": round(ms / 1000 / audio_s, 2) if audio_s else None,
        }
    )
    print(f"{'cold' if i == 0 else 'warm'}  {ms:7.0f} ms  {audio_s:4.1f} s audio  {text}")
    display(Audio(wavs[0], rate=sr))

## 6. Reuse the saved impression

Simulates an app restart: load `voice_prompt.pt` instead of re-analyzing the clip.

In [ ]:
reloaded = torch.load("voice_prompt.pt", weights_only=False)
t = time.perf_counter()
wavs, sr = tts.generate_voice_clone(
    text="This voice was loaded from disk.", language=LANGUAGE, voice_clone_prompt=reloaded
)
print(f"{(time.perf_counter() - t) * 1000:.0f} ms from the saved prompt")
display(Audio(wavs[0], rate=sr))

## 7. Results

Copy the printed row into `docs/latency.md`. Targets from `AGENTS(GH).md`: instant phrases < 1 s (they're pre-synthesized, so generation time only matters once) and suggested replies 2–3 s after the speaker stops.

A `realtime_factor` under 1.0 means audio is generated faster than it plays.

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
warm = df.iloc[1:] if len(df) > 1 else df
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
size = MODEL_ID.split("-")[-2]
print(f"model: {MODEL_ID} | device: {gpu} | dtype: {dtype} | load: {load_s:.1f} s | clone: {clone_ms:.0f} ms")
print(f"warm median: {warm.gen_ms.median():.0f} ms per line | median realtime factor: {warm.realtime_factor.median():.2f}")
display(df)

print()
print("Row for docs/latency.md:")
print(
    f"| {time.strftime('%Y-%m-%d')} | {gpu} | `/tts` Qwen3-TTS {size} ({len(SAMPLE_LINES)} lines) "
    f"| {df.gen_ms.iloc[0]} | {warm.gen_ms.median():.0f} | clone {clone_ms:.0f} ms, load {load_s:.0f} s |"
)

In [ ]:
# Download the outputs, then delete the runtime so the voice clip doesn't linger.
import shutil

shutil.make_archive("qwen3_tts_spike", "zip", "out")
try:
    from google.colab import files

    files.download("qwen3_tts_spike.zip")
except ImportError:
    print("Saved qwen3_tts_spike.zip in the working directory.")